# Landlab and PyVista: further examples

*(Greg Tucker, University of Colorado Boulder, USA, Jul-Aug 2025)*

This notebook presents more extensive examples of visualizing Landlab output using PyVista. It builds on two previous notebooks, which should be reviewed first: *Translating a Landlab RasterModelGrid into a PyVista StructuredGrid for visualization* and *Translating a Landlab non-raster grid into PyVista for visualization*.

## Example of a listric normal fault

This example illustrates the overlay of two meshes, representing different surfaces. The example uses Landlab's `ListricKinematicExtender` component to simulate progressive slip on a listric ("spoon-shaped") extensional fault. The visualization shows both the topographic surface and the fault plane. The example also illustrates the use of a Landlab `HexModelGrid`.

For imports, we need Pvista as well as the `grid_to_pv` function in `llpvtools` (the package in which this notebook originates).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from landlab import HexModelGrid, imshow_grid
from landlab.components import ListricKinematicExtender
from llpvtools import grid_to_pv

### Set up and run a model with tectonic extension

Here, we set up and run a model that will create a topographic surface as well as a fault-plane surface, and generate some offset along the fault.

First, we set various parameters. We'll keep the grid relatively small for the sake of illustration.

In [ ]:
# parameters
nrows = 31  # number of node rows
ncols = 51  # number of node columns
dx = 1000.0  # grid spacing, m
nsteps = 20  # number of iterations
dt = 2.5e5  # time step, y
extension_rate = 0.001  # rate of tectonic extension, m/y
detachment_depth = 10000.0  # depth at which the fault plane becomes horizontal, m
fault_dip = 60.0  # fault dip angle, degrees
fault_x0 = (
    10000.0  # initial x coordinate of fault (at y=0), in m from left side of model
)
fault_strike = 60.0  # fault strike, degrees ccw from east
period = 15000.0  # period of sinusoidal variations in initial topography, m
ampl = 500.0  # amplitude of variations, m

We create a Landlab `HexModelGrid`, and give it an elevation field with sinusoidal "bumps". We then instantiate a `ListricKinematicExtender` component (see the [documentation](https://landlab.csdms.io/tutorials/tectonics/listric_kinematic_extender.html) for information about its parameters and how it works).

In [ ]:
# Create grid and elevation field
grid = HexModelGrid((nrows, ncols), spacing=dx, node_layout="rect")

# The elevation field has a "bumpy" surface for the sake of illustration
elev = grid.add_zeros("topographic__elevation", at="node")
elev[:] = (
    ampl
    * np.sin(2 * np.pi * grid.x_of_node / period)
    * np.sin(2 * np.pi * grid.y_of_node / period)
)

# Instantiate component
extender = ListricKinematicExtender(
    grid,
    extension_rate_x=extension_rate,
    extension_rate_y=0.0,
    fault_dip=fault_dip,
    fault_strike=fault_strike,
    fault_x0=fault_x0,
    fault_y0=0.0,
    detachment_depth=detachment_depth,
)

Run the component forward in time, such that the hangingwall moves down and apart from the footwall block.

In [ ]:
# Run
for i in range(nsteps):
    extender.run_one_step(dt)

### Visualization

Here we translate the Landlab grid into a Pyvista mesh and display it.

First, generate the mesh for the topography:

In [ ]:
# Translate the Landlab HexModelGrid and fields into a Pyvista mesh, using topography for the z coordinate
# (here we only need the node-link-patch mesh, so we ignore the corresponding corner-face-cell mesh)
topomeshn, _ = grid_to_pv(grid, field_for_node_z=elev, field_for_corner_z=0.0)

Now, a mesh for the fault plane:

In [ ]:
# Create a second mesh, this time using the fault plane elevation as the z coordinate (again ignoring
# the dual corner-face-cell mesh)
faultmeshn, _ = grid_to_pv(
    grid, field_for_node_z="fault_plane__elevation", field_for_corner_z=0.0
)

Create a Pyvista `Plotter` object and add both meshes, then `show` the result:

In [ ]:
# Color the topography mesh by elevation, and fault plane mesh by fault-plane elevation
topomeshn.set_active_scalars("topographic__elevation")
faultmeshn.set_active_scalars("fault_plane__elevation")
fault_elev = grid.at_node["fault_plane__elevation"]

# Set color scale range for topography so it matches the range for the fault plane
cmin = np.amin(fault_elev)
cmax = np.amax(fault_elev)

pvp = pv.Plotter()
pvp.add_mesh(topomeshn, opacity=0.6, clim=[cmin, cmax], show_edges=True, cmap="autumn")
pvp.add_mesh(faultmeshn, opacity=0.6, show_edges=True, cmap="autumn")
pvp.show()

## Growing and eroding a carbonate platform

The next example illustrates the use of a [`FramedVoronoiGrid`](https://landlab.csdms.io/generated/api/landlab.grid.framed_voronoi.html#landlab.grid.framed_voronoi.FramedVoronoiGrid) (see the tutorial on [Diverse Grid Classes](https://landlab.csdms.io/tutorials/grids/diverse_grid_classes.html#FramedVoronoiGrid)), with an added layer to represent current sea level.

The example simulates carbonate platform generation on a sloping margin subject to sinusoidal sea-level oscillation and background tectonic uplift of the margin. It uses the Landlab [`CarbonateProducer`](https://landlab.csdms.io/generated/api/landlab.components.carbonate.carbonate_producer.html#landlab.components.carbonate.carbonate_producer.CarbonateProducer) component (see the [CarbonateProducer](https://landlab.csdms.io/tutorials/carbonates/carbonate_producer.html) tutorial).

In [ ]:
from landlab import FramedVoronoiGrid
from landlab.components import CarbonateProducer
from landlab.plot import plot_layers

In [ ]:
# Parameters and settings
nrows = 21  # number of grid rows
ncols = 81  # number of grid columns
dx = 1000.0  # node spacing, m
sl_range = 65.0  # range of sea-level variation, m
sl_period = 100000.0  # period of sea-level variation, y
run_duration = 300000.0  # duration of run, y
dt = 100.0  # time-step size, y
topo_slope = 0.01  # initial slope of the topography/bathymetry, m/m
initial_shoreline_pos = 25000.0  # starting position of the shoreline, m
uplift_rate = 0.001  # rate of tectonic uplift, m/y
ve = 10.0  # vertical exaggeration factor for plotting

In [ ]:
# Derived parameters
num_steps = int(run_duration / dt)
sin_fac = 2.0 * np.pi / sl_period  # factor for sine calculation of sea-level

In [ ]:
# Grid and fields

# Create a grid object
grid = FramedVoronoiGrid((nrows, ncols), xy_spacing=dx)

# add fields for sea level, surface elevation, carbonate thickness
sea_level = grid.add_field("sea_level__elevation", 0.0, at="grid")
base_elev = grid.add_zeros("basement__elevation", at="node")
base_elev[:] = topo_slope * (initial_shoreline_pos - grid.x_of_node)
elev = grid.add_zeros("topographic__elevation", at="node")
carbo_thickness = grid.add_zeros("carbonate_thickness", at="node")

# initialize the elevation to the basement elevation
elev[:] = base_elev

In [ ]:
# Instantiate component
cp = CarbonateProducer(grid)

In [ ]:
# RUN
for i in range(num_steps):
    cp.sea_level = sl_range * np.sin(sin_fac * i * dt)  # update sea level
    cp.produce_carbonate(dt)  # produce some carbonate
    elev[:] += uplift_rate * dt  # add tectonic uplift

Now that the model has run, use PyVista to display the results. First, apply some vertical exaggeration to the $z$ coordinate (i.e., surface elevation). Then, create an additional PyVista mesh to represent the sea surface. Finally, visualize the topography and sea level together.

In [ ]:
# Create a version of the elevation array with vertical exaggeration, for plotting
elev_ve = elev * ve

# Generate a pair of PyVista meshes
meshn, meshc = grid_to_pv(grid, field_for_node_z=elev_ve, field_for_corner_z=0.0)

In [ ]:
# Create a sea-level mesh
xmin = np.amin(grid.x_of_node)
xmax = np.amax(grid.x_of_node)
ymin = np.amin(grid.y_of_node)
ymax = np.amax(grid.y_of_node)
x = np.array([xmin, xmax, xmin, xmax])
y = np.array([ymin, ymin, ymax, ymax])
z = np.array([sea_level, sea_level, sea_level, sea_level])

pts = np.c_[x.reshape(-1), y.reshape(-1), z.reshape(-1)]
pd = pv.PolyData(pts)
slsurf = pd.delaunay_2d()

slsurf.plot()

Create a PyVista `Plotter` object that has our topographic mesh plus the sea-level mesh, and display them.

In [ ]:
pvp2 = pv.Plotter()
pvp2.add_mesh(meshn)
pvp2.add_mesh(slsurf)
pvp2.show()

The irregularity in the terrain reflects the semi-random geometry of the `FramedVoronoiGrid`. Here's what the grid looks like:

In [ ]:
meshn.plot(show_edges=True)

### Please share!

If you end up using `llpvtools` to create interesting visualizations, please share it on the [CSDMS Forum](https://forum.csdms.io/) or drop us a line at csdms@colorado.edu!

#### To cite this tool and other CSDMS products:

Tucker, G.E., Hutton, E.W., Piper, M.D., Campforts, B., Gan, T., Barnhart, K.R., Kettner, A.J., Overeem, I., Peckham, S.D., McCready, L. and Syvitski, J., 2022. CSDMS: a community platform for numerical modeling of Earth surface processes. Geoscientific Model Development, 15(4).

To cite Landlab specifically:

Hobley, D.E., Adams, J.M., Nudurupati, S.S., Hutton, E.W., Gasparini, N.M., Istanbulluoglu, E. and Tucker, G.E., 2017. Creative computing with Landlab: an open-source toolkit for building, coupling, and exploring two-dimensional numerical models of Earth-surface dynamics. Earth Surface Dynamics, 5(1), pp.21-46.

Barnhart, K.R., Hutton, E.W., Tucker, G.E., Gasparini, N.M., Istanbulluoglu, E., Hobley, D.E., Lyons, N.J., Mouchene, M., Nudurupati, S.S., Adams, J.M. and Bandaragoda, C., 2020. Landlab v2. 0: A software package for Earth surface dynamics. Earth Surface Dynamics, 8(2), pp.379-397.